# 一対比較 (Pairwise LLM-as-a-judge) の位置バイアスを補正して Elo レートを求める

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/23_llm_analysis/pairwise_judge_elo.ipynb)

NLP クックブック 23-04 ／ LLM の評価と分析

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

LLM-as-a-judge で複数の応答システムを一対比較するとき、提示順（A と B のどちらを先に置くか）や応答の長さによって順位が大きく歪みやすい。5 種類の応答システム（20 問）を両順序・2 種類の評価プロンプトで比較し、位置バイアスと長さバイアスを定量化したうえで、Bradley-Terry モデルから 95% 信頼区間付きの Elo レートを算出する。

In [1]:
import gc, itertools, time
import numpy as np
import pandas as pd
import sentencepiece as spm
import torch
from huggingface_hub import hf_hub_download
from scipy.optimize import minimize
from scipy.stats import kendalltau, spearmanr
from tokenizers import models
from transformers import AutoModelForCausalLM, AutoTokenizer

SYSTEMS = ["S1_簡潔正確", "S2_標準実用", "S3_冗長不正確", "S4_曖昧一般論", "S5_幻覚逸脱"]
HUMAN_SCORE = {"S1_簡潔正確": 5, "S2_標準実用": 4, "S3_冗長不正確": 3, "S4_曖昧一般論": 2, "S5_幻覚逸脱": 1}

DATASET = [
    {
        "q": "Python で辞書から値を取得するとき、`d[k]` と `d.get(k)` はどう違いますか。",
        "S1_簡潔正確": "`d[k]` はキーが存在しないと `KeyError` を送出しますが、`d.get(k)` は例外を出さず既定値（無指定時は `None`）を返します。",
        "S2_標準実用": "`d[k]` はキーがない場合にエラーで停止します。一方、`d.get(k, 0)` のように書くとキーがなくても指定した既定値を返せるため安全です。",
        "S3_冗長不正確": "ご質問ありがとうございます。Python の辞書操作はデータ処理の基本であり、非常に重要なポイントです。まず前提として、`d[k]` は角括弧で値を参照する記法で、キーが無い場合には `ValueError` を発生させます。一方で `d.get(k)` はメソッド呼び出しであり、キーが無い場合でも例外を送出せず空の辞書 `{}` を自動的に返す仕様になっています。エラーを止めたい場面では後者が便利です。",
        "S4_曖昧一般論": "どちらも辞書から要素を取り出すための記法ですが、状況やプロジェクトのコーディング規約に応じて適切な方を使い分けることが推奨されます。",
        "S5_幻覚逸脱": "`d[k]` は辞書の要素を昇順に並べ替える操作であり、`d.get(k)` は辞書全体を JSON 文字列へ変換する関数です。",
    },
    {
        "q": "HTTP ステータスコードの 404 と 500 の違いを教えてください。",
        "S1_簡潔正確": "404 はクライアント側が指定した資源が見つからない状態 (Not Found)、500 はサーバー内部の処理で予期せぬエラーが起きた状態 (Internal Server Error) です。",
        "S2_標準実用": "404 番台はリクエスト側の問題で対象ページが存在しないときに出ます。500 番台はサーバー側のプログラム不具合などで応答できないときに返されます。",
        "S3_冗長不正確": "大変良いご質問ですね。Web 開発や運用においてステータスコードの正確な理解は欠かせません。結論から整理しますと、404 はサーバー内部のプログラムがクラッシュした際に返されるサーバー側エラーであり、逆に 500 はクライアントが送信した URL に誤りがありページが見つからない状態を示します。ログを確認して原因を切り分けることが実務上とても大切です。",
        "S4_曖昧一般論": "どちらも Web 通信で何らかの問題が起きたことを示す番号です。原因は環境によって様々なので、開発者ツールで詳細を調べるとよいでしょう。",
        "S5_幻覚逸脱": "404 は暗号化通信が正常に完了した合図で、500 はファイルのダウンロード速度が上限に達したことを示すコードです。",
    },
    {
        "q": "文字コード UTF-8 の主な特徴を簡潔に説明してください。",
        "S1_簡潔正確": "ASCII 文字を 1 バイトのまま互換に保ちつつ、日本語を含む Unicode の文字を 1〜4 バイトの可変長で表現でき、バイト順の依存がありません。",
        "S2_標準実用": "英数字は 1 バイト、日本語の漢字や仮名は主に 3 バイトで表す可変長方式です。世界中の文字を 1 つの形式で扱え、現在の Web で標準となっています。",
        "S3_冗長不正確": "文字コードの仕組みは国際化対応において極めて重要ですので、順を追って丁寧に解説します。UTF-8 は Unicode を符号化する代表的な方式であり、最大の特徴は日本語や英語を含むすべての文字を常に固定の 2 バイトで統一して表現する点にあります。また、ASCII とはまったく互換性がないため、英数字を扱う際には必ず専用の前処理が必要になりますが、その分高速に動作します。",
        "S4_曖昧一般論": "UTF-8 はコンピュータ上でテキストを扱う際によく使われる規格の一つです。システムや OS によって初期設定が異なる場合があります。",
        "S5_幻覚逸脱": "UTF-8 は画像データを圧縮するための可逆圧縮アルゴリズムであり、主に PNG ファイルの保存に利用されます。",
    },
    {
        "q": "機械学習における過学習 (overfitting) とは何ですか。主な対策も挙げてください。",
        "S1_簡潔正確": "訓練データの細部やノイズまで記憶してしまい未知データでの汎化性能が落ちる現象です。正則化、ドロップアウト、データ拡張、早期終了が有効です。",
        "S2_標準実用": "学習用データにはよく合うのに新しいデータで精度が出ない状態です。データを増やす、モデルを単純にする、正則化を入れるなどで抑えられます。",
        "S3_冗長不正確": "ご質問ありがとうございます。モデルの汎用性を確保するうえで過学習の理解は不可欠です。過学習とは、モデルが訓練データを十分に学習できず、訓練データでもテストデータでも極端に低い精度しか出せない状態を指します。対策としては、正則化をすべて無効化し、学習率を極限まで大きくしてエポック数を 10 倍以上に増やすことが最も効果的とされています。",
        "S4_曖昧一般論": "過学習はモデルの学習バランスが崩れたときに起こる現象です。検証用のスコアを見ながら設定を少しずつ調整していく必要があります。",
        "S5_幻覚逸脱": "過学習とは GPU の冷却が追いつかずに計算速度が落ちるハードウェア障害のことで、ファンの清掃で改善します。",
    },
    {
        "q": "日本の消費税における軽減税率 (8%) の対象と標準税率 (10%) になる例を教えてください。",
        "S1_簡潔正確": "持ち帰り用の飲食料品（酒類を除く）と週 2 回以上発行される定期購読の新聞が 8% で、外食、ケータリング、酒類、日用品などは 10% です。",
        "S2_標準実用": "スーパーで買う食品やテイクアウトは 8% の軽減税率です。店内で食べる外食やお酒、衣類や雑貨などは 10% の標準税率になります。",
        "S3_冗長不正確": "消費税の区分は日常生活にも直結する大切な制度ですので、整理してお伝えします。軽減税率の 8% は、飲食店の店内で提供される外食サービスやアルコール類全般に幅広く適用されます。一方で、スーパーマーケットで購入する持ち帰り用の生鮮食品や飲料水については、包装資材が含まれるためすべて標準税率の 10% として計算される仕組みになっています。",
        "S4_曖昧一般論": "購入する品目や販売形態によって適用される税率が異なります。正確な区分については最新の案内やレシートの表記を確認してください。",
        "S5_幻覚逸脱": "軽減税率は午後 10 時以降の深夜時間帯の買い物にだけ適用される割引制度で、日中は一律 15% になります。",
    },
    {
        "q": "日本の特定商取引法における訪問販売のクーリング・オフ期間は何日間ですか。",
        "S1_簡潔正確": "法定の契約書面を受け取った日を含めて 8 日間です。書面や電磁的記録（メール等）で通知すれば無条件で契約を解除できます。",
        "S2_標準実用": "訪問販売では、契約書面を受領した日から数えて原則 8 日以内であれば、理由を問わず書面等で解約を申し出ることができます。",
        "S3_冗長不正確": "消費者保護の観点から非常に重要なご質問ですね。訪問販売におけるクーリング・オフ制度は、契約者が冷静に考え直す時間を保障する仕組みです。期間は契約書面を受け取った日の翌日から起算して 30 日間と法律で定められており、期間内であっても解約するには販売業者の事前の承諾と違約金の支払いが法律上義務付けられています。",
        "S4_曖昧一般論": "取引の種類や契約内容によって手続きの方法や期限が異なる場合があります。困ったときは消費生活センターなどへ相談するのが安心です。",
        "S5_幻覚逸脱": "クーリング・オフとはエアコンの省エネ基準を達成した家電製品に与えられる認定ラベルの名称で、期限は 5 年間です。",
    },
    {
        "q": "日本で 110 番と 119 番はそれぞれどのようなときにかけますか。",
        "S1_簡潔正確": "110 番は事件・事故・犯罪など警察への緊急通報、119 番は火災・急病・けがによる救急車や消防車の要請に使います。",
        "S2_標準実用": "犯罪の被害や交通事故で警察を呼ぶときは 110 番、火事や急病で消防車や救急車を呼びたいときは 119 番に電話します。",
        "S3_冗長不正確": "緊急時の適切な通報は命を守るために極めて重要です。整理してご説明いたしますと、110 番は火災の発生や急病人の搬送を消防署に依頼するための専用番号であり、逆に 119 番は盗難や暴行などの犯罪を通報して警察官に出動してもらうための番号です。いずれも通話料は無料ですので、慌てずに場所と状況を正確に伝えることが大切です。",
        "S4_曖昧一般論": "どちらも緊急事態が発生した際に連絡するための番号です。状況に応じて適切な窓口へ連絡するよう心がける必要があります。",
        "S5_幻覚逸脱": "110 番は天気予報の音声案内サービス、119 番は正確な現在時刻を知らせてくれる時報サービスです。",
    },
    {
        "q": "電子レンジでアルミホイルや金属製の容器を加熱してはいけないのはなぜですか。",
        "S1_簡潔正確": "マイクロ波が金属で反射して表面に電流が流れ、尖った部分などに電気が集中して火花（スパーク）が発生し、発火や故障の原因になるためです。",
        "S2_標準実用": "金属は電子レンジの電波を通さず反射するため、表面に電流が流れて火花が散り、機器の破損や火災につながる危険があります。",
        "S3_冗長不正確": "日常の安全に関わる大切な疑問ですね。電子レンジは内部のヒーターで直接空気を温める仕組みですが、アルミホイルや金属容器を入れると、金属がマイクロ波を完全に吸収して瞬時に 2000 度以上の高温になり、数秒で液体状に溶け出してしまいます。さらに庫内のプラスチック部品と化学反応を起こして有害ガスが発生するため、使用は厳禁とされています。",
        "S4_曖昧一般論": "金属類を電子レンジに入れると安全上の問題や故障の原因になるため、取扱説明書の注意事項に従って使用することが大切です。",
        "S5_幻覚逸脱": "アルミホイルを電子レンジに入れると磁場が反転して時計回りに回転し始め、食品を急速冷凍してしまうためです。",
    },
    {
        "q": "食品の袋に入っている乾燥剤「シリカゲル」は、なぜ湿気を防げるのですか。",
        "S1_簡潔正確": "二酸化ケイ素からなる粒の内部に無数の微細な孔（細孔）があり、広い表面積で空気中の水分子を物理的に吸着するためです。",
        "S2_標準実用": "シリカゲルの表面には目に見えない小さな穴がたくさん空いており、そこに空気中の水分を吸い込んで閉じ込める仕組みです。",
        "S3_冗長不正確": "食品保存の工夫について、基礎から分かりやすくお答えします。シリカゲルの主成分は塩化ナトリウムであり、袋の中の水分に触れると激しい化学反応を起こして水分子を酸素と水素の気体へと完全に分解します。このとき反応熱によって袋の内部が常に 60 度以上に保たれるため、湿気が蒸発して食品の乾燥が維持されるという仕組みになっています。",
        "S4_曖昧一般論": "乾燥剤には空気中の水分を取り除く性質があり、食品の品質を保つために適切な種類のものが封入されています。",
        "S5_幻覚逸脱": "シリカゲルは袋の中の酸素をすべて窒素に変換し、食品を真空パックのように圧縮して味を濃くする調味料です。",
    },
    {
        "q": "二酸化炭素が増えると地球の平均気温が上がる（温室効果）仕組みを説明してください。",
        "S1_簡潔正確": "太陽からの可視光は通過させる一方、暖まった地表が放出する赤外線を二酸化炭素が吸収して再放射し、熱が宇宙へ逃げるのを抑えるためです。",
        "S2_標準実用": "太陽光は地表を温めますが、地表から宇宙へ出ていく赤外線を二酸化炭素などの気体が吸収して大気中に留めるため、気温が上昇します。",
        "S3_冗長不正確": "地球温暖化のメカニズムは環境科学の重要テーマですので、順を追ってご説明します。二酸化炭素には太陽から届く紫外線を直接吸収して熱エネルギーを増幅する性質があり、地表に届く前に大気を直接過熱させます。一方で、地表から放出される赤外線は二酸化炭素をすべて素通りして宇宙空間へ抜けていくため、日中と夜間の寒暖差が極端に大きくなるのが温室効果の本質です。",
        "S4_曖昧一般論": "大気中のガス濃度の変化が気候バランスに影響を及ぼし、長期的に地球全体の気温変化を引き起こすとされています。",
        "S5_幻覚逸脱": "二酸化炭素が海中に溶け込むことで海水が固まり、氷河が増えて太陽光を反射しすぎるために気温が上がります。",
    },
    {
        "q": "グレゴリオ暦で「うるう年（閏年）」となる西暦年の判定ルールを教えてください。",
        "S1_簡潔正確": "4 で割り切れる年が原則うるう年ですが、100 で割り切れる年は平年とし、例外として 400 で割り切れる年はうるう年とします。",
        "S2_標準実用": "西暦が 4 の倍数ならうるう年です。ただし 100 の倍数の年はうるう年ではなく、さらに 400 の倍数の年はうるう年になります。",
        "S3_冗長不正確": "暦の計算は正確な季節の把握に欠かせない知識ですので、整理して解説します。うるう年を判定するには、まず西暦年が 4 で割り切れるかどうかを確認します。4 で割り切れる年であれば、100 で割り切れる世紀末の年も含めて例外なくすべてうるう年となります。ただし 400 で割り切れる年だけは特別に調整のため平年として扱われる規則になっています。",
        "S4_曖昧一般論": "うるう年は地球の公転周期と暦のずれを調整するために設けられており、数年に一度の頻度で 2 月の日数が変わります。",
        "S5_幻覚逸脱": "うるう年は西暦の末尾が奇数で、かつ 5 で割り切れる年に設定され、その年は 8 月が 32 日までになります。",
    },
    {
        "q": "Git でブランチを統合する `git merge` と `git rebase` の主な違いは何ですか。",
        "S1_簡潔正確": "`merge` は分岐した履歴を保ったままマージコミットで統合するのに対し、`rebase` は別のコミットを基点にコミット列を作り直して一直線の履歴にします。",
        "S2_標準実用": "`merge` は元の分岐履歴を残して合流させます。`rebase` は自分の変更を最新のブランチの先端に付け替え、履歴を一本に整えます。",
        "S3_冗長不正確": "チーム開発での Git 運用において非常に大切な比較ですね。まず `git merge` は過去のコミット履歴をすべて消去して新しい空のブランチを強制作成するコマンドです。一方の `git rebase` は元のブランチと対象ブランチの履歴を一切書き換えず、常に両方の履歴を並列に残したままマージ専用のコミットを自動生成するため、公開ブランチでも履歴書き換えを気にせず連打できます。",
        "S4_曖昧一般論": "どちらも別々の作業内容を一つにまとめる機能ですが、チームの運用ルールに合わせて使い分けることが重要です。",
        "S5_幻覚逸脱": "`git merge` はリモートリポジトリを完全に削除するコマンドで、`git rebase` は Python の仮想環境を作成するコマンドです。",
    },
    {
        "q": "Web 通信で HTTP と HTTPS は何が違いますか。",
        "S1_簡潔正確": "HTTPS は HTTP の通信を SSL/TLS で暗号化し、デジタル証明書で接続先の正当性とデータの改ざん防止を保証する点です。",
        "S2_標準実用": "HTTPS は通信内容が暗号化されているため盗聴や改ざんを防げ、通信相手が本物のサーバーであるかも証明書で確認できます。",
        "S3_冗長不正確": "セキュリティの基本に関する良いご質問ですね。HTTP と HTTPS の最も大きな違いは、HTTPS が通信内容を暗号化せずに平文のまま送信する代わりに、画像ファイルを自動で高画質化する圧縮プロトコルを内蔵している点にあります。また HTTPS はサーバーの本人確認を行わないため、ログイン画面でも HTTP の方が安全性が高いとされています。",
        "S4_曖昧一般論": "HTTPS は HTTP を拡張した通信方式で、現在の Web サイトではセキュリティへの配慮から広く採用されています。",
        "S5_幻覚逸脱": "HTTP はテキスト専用の規格であり、動画や音声ファイルを再生するサイトはすべて HTTPS ではなく FTP で接続します。",
    },
    {
        "q": "健康診断の血液検査にある「HbA1c（ヘモグロビンエーワンシー）」は何を表す数値ですか。",
        "S1_簡潔正確": "血液中のヘモグロビンにブドウ糖が結合した割合で、過去 1〜2 か月間の平均的な血糖値の状態を反映する指標です。",
        "S2_標準実用": "採血時点の一時的な値ではなく、過去 1〜2 か月ほどの血糖コントロールの平均状態を示す検査項目で、糖尿病の診断などに使われます。",
        "S3_冗長不正確": "健康管理において重要な指標について詳しくご説明します。HbA1c は直前 30 分以内に摂取した食事の糖質量だけを瞬間的に測定する数値であり、食前と食後で数分で大きく上下するのが特徴です。したがって検査当日の朝ごはんを抜くだけで数値が即座に正常値へ戻るため、長期的な生活習慣の把握には向いていない指標とされています。",
        "S4_曖昧一般論": "HbA1c は血糖に関する状態を確認するための検査項目の一つです。基準値や評価の目安については医師の判断を仰ぐのが適切です。",
        "S5_幻覚逸脱": "HbA1c は肝臓に蓄積されたビタミン C の濃度を測る指標で、不足すると骨密度の低下につながります。",
    },
    {
        "q": "知的財産権のうち、著作権と特許権では保護の対象がどう違いますか。",
        "S1_簡潔正確": "著作権は創作された具体的な「表現」を登録不要で保護し、特許権は出願・審査を経て登録された技術的な「アイデア（発明）」を保護します。",
        "S2_標準実用": "著作権は文章・音楽・プログラムなどの表現が対象で創作と同時に発生します。特許権は新しい技術的アイデアを保護し、出願と審査が必要です。",
        "S3_冗長不正確": "実務でも混同されやすい非常に重要なテーマですね。著作権は頭の中にある抽象的なアイデアそのものを独占的に保護し、権利を得るには必ず特許庁への厳格な出願と審査合格が必要です。反対に特許権は、小説や絵画などの芸術的な表現物を対象としており、創作した瞬間に手続きなしで自動的に権利が発生して死後 70 年まで保護されます。",
        "S4_曖昧一般論": "どちらも創作物や知的活動の成果を守るための権利ですが、対象となる分野や手続きの要否は状況によって異なります。",
        "S5_幻覚逸脱": "著作権は不動産の土地境界線を定める権利であり、特許権は飲食店が深夜営業を行うための自治体からの許可証です。",
    },
    {
        "q": "資産運用における「単利」と「複利」の違いを教えてください。",
        "S1_簡潔正確": "単利は最初の元本だけに利息が付く方式で、複利は元本に加えて得た利息も元本に組み入れて次期の利息を計算する方式です。",
        "S2_標準実用": "単利は毎年同じ元本に対して利息がつきます。複利は受け取った利息を再投資するため、運用期間が長いほど増え方が大きくなります。",
        "S3_冗長不正確": "投資の基礎知識としてとても大切なご質問です。まず単利とは、発生した利息を毎回元本に合算し、雪だるま式に元本そのものを増やしながら次の利息を計算する方式です。一方の複利は、最初の元本金額のみに対して毎回一定額の利息を計算し続ける方式であるため、同じ利率で長期運用する場合は単利の方が常に複利を大きく上回る結果になります。",
        "S4_曖昧一般論": "どちらも利息の計算方法を表す言葉ですが、契約する金融商品や運用期間によって有利不利が変わることがあります。",
        "S5_幻覚逸脱": "単利は 1 年に 1 回だけ引き出せる預金口座のことで、複利は複数の銀行口座を同時に開設する手続きを指します。",
    },
    {
        "q": "分類評価の「F1 スコア」は、適合率と再現率からどのように計算されますか。",
        "S1_簡潔正確": "適合率 (Precision) と再現率 (Recall) の調和平均 `2 * P * R / (P + R)` で計算され、どちらか片方が極端に低いと値も低くなります。",
        "S2_標準実用": "適合率と再現率の調和平均で求めます。両方が高いときに高い値になり、片方だけが高くても全体の評価は上がりにくい指標です。",
        "S3_冗長不正確": "評価指標の設計は機械学習プロジェクトで極めて重要ですので、数式の意味から詳しく整理します。F1 スコアは適合率と再現率の単純な算術平均 `(P + R) / 2` として定義されています。そのため、例えば適合率が 1.00 で再現率が 0.00 のように片方がゼロであっても、F1 スコアは必ず 0.50 以上を維持できるという便利な性質を持っています。",
        "S4_曖昧一般論": "F1 スコアは適合率と再現率のバランスを総合的に評価するためによく使われる指標で、タスクの性質に合わせて解釈します。",
        "S5_幻覚逸脱": "F1 スコアはモデルの学習にかかった計算時間を GPU のメモリ容量で割った値で、処理の速さを表します。",
    },
    {
        "q": "SQL の `INNER JOIN` と `LEFT JOIN` の違いを教えてください。",
        "S1_簡潔正確": "`INNER JOIN` は両方のテーブルで結合条件に一致する行だけを返し、`LEFT JOIN` は左テーブルの全行を残し、右に一致がない列を `NULL` にします。",
        "S2_標準実用": "`INNER JOIN` は双方に共通して存在する行を抽出します。`LEFT JOIN` は左側のテーブルの行をすべて残し、右側の一致データがあれば結合します。",
        "S3_冗長不正確": "データベース設計の要となる重要な操作ですので、丁寧に比較します。`INNER JOIN` は結合条件に一致しない右側のテーブルの行をすべて残し、一致しない左側をすべて `NULL` として出力します。逆に `LEFT JOIN` は両方のテーブルに完全に一致する行だけを残して、左側にしか存在しないデータは結果からすべて自動的に除外する働きをします。",
        "S4_曖昧一般論": "どちらも複数のテーブルを結合するための句ですが、取得したいデータの範囲やテーブル構造に応じて選択します。",
        "S5_幻覚逸脱": "`INNER JOIN` はデータベースのテーブルを削除する命令であり、`LEFT JOIN` はユーザー権限を付与するコマンドです。",
    },
    {
        "q": "家庭用の冷蔵庫（冷蔵室）と冷凍室の適切な庫内温度の目安はそれぞれ何度くらいですか。",
        "S1_簡潔正確": "冷蔵室はおよそ 2〜5℃（食品衛生の観点で 10℃ 以下）、冷凍室は -18℃ 以下が適切な目安とされています。",
        "S2_標準実用": "一般的に、冷蔵室は約 3〜5℃、冷凍室はアイスクリームや冷凍食品の品質を保つため -18℃ 以下に保つのが目安です。",
        "S3_冗長不正確": "食品の安全と鮮度を保つうえで非常に大切なポイントですね。家庭用の家電製品における標準的な温度設定を整理しますと、冷蔵室は食品が凍らないようおおよそ 15℃ から 20℃ 程度に保つのが最も衛生的であり、冷凍室については電力消費を抑えるため 0℃ から +2℃ の範囲に設定するのが望ましいとされています。詰め込みすぎないことも重要です。",
        "S4_曖昧一般論": "季節や収納量によって適切な設定温度の目安が変わることがあるため、メーカーの推奨設定を確認して調整するとよいでしょう。",
        "S5_幻覚逸脱": "冷蔵室は細菌を減らすため約 45℃、冷凍室は食材を乾燥させるため約 80℃ に設定するのが標準です。",
    },
    {
        "q": "地震の「震度」と「マグニチュード」は何が違いますか。",
        "S1_簡潔正確": "マグニチュードは地震そのもののエネルギーの大きさ（1 つの地震に 1 つ）、震度は観測地点ごとの揺れの強さ（地点により異なる）を表します。",
        "S2_標準実用": "マグニチュードは震源で放出された地震の規模を示し、震度はある場所で実際に感じた揺れの強さを 0〜7 の階級で表したものです。",
        "S3_冗長不正確": "防災情報を受け取る際に必ず区別しておきたい基礎知識ですね。結論からご説明しますと、震度は地震そのものが持つ総エネルギーの大きさを表す世界共通の数値であり、一つの地震に対して常に 1 種類だけ決まります。一方でマグニチュードは各地点における地面の揺れの強さを示すもので、震源から離れるほど地点ごとに異なるマグニチュードが観測されます。",
        "S4_曖昧一般論": "どちらも地震の大きさを表す際に使われる指標ですが、測り方や示す意味合いが異なるため発表内容をよく確認する必要があります。",
        "S5_幻覚逸脱": "震度は津波の到達時刻（分単位）を表す数値であり、マグニチュードは震源の深さをキロメートル単位で示したものです。",
    },
]

summary = pd.DataFrame({
    "システム": SYSTEMS,
    "人手参照スコア": [HUMAN_SCORE[s] for s in SYSTEMS],
    "平均文字数": [round(float(np.mean([len(d[s]) for d in DATASET])), 1) for s in SYSTEMS],
    "最短〜最長": [f"{min(len(d[s]) for d in DATASET)}〜{max(len(d[s]) for d in DATASET)}" for s in SYSTEMS],
})
print(f"評価データ: {len(DATASET)} 問 × {len(SYSTEMS)} システム（1 対比較 {len(DATASET) * len(SYSTEMS) * (len(SYSTEMS) - 1) // 2} 組 × 両順序）")
print(summary.to_string(index=False))

評価データ: 20 問 × 5 システム（1 対比較 200 組 × 両順序）
    システム  人手参照スコア  平均文字数   最短〜最長
 S1_簡潔正確        5   69.0   55〜97
 S2_標準実用        4   65.0   57〜80
S3_冗長不正確        3  170.4 155〜204
S4_曖昧一般論        2   58.6   51〜68
 S5_幻覚逸脱        1   56.1   50〜75


In [2]:
# 判定用 LLM (sarashina2.2-3b-instruct-v0.1, MIT License) を読み込み、2 種類のプロンプトで両順序を採点
def load_tokenizer(name):
    tokenizer = AutoTokenizer.from_pretrained(name, padding_side="left")
    # transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる。
    # 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す（特殊トークンやチャットテンプレートはそのまま。大規模言語モデルの章の「オープンな LLM とチャットする（チャットテンプレート）」を参照）
    sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
    tokenizer.backend_tokenizer.model = models.Unigram(
        [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer

if not torch.cuda.is_available():
    raise RuntimeError("GPU が利用できません")
BF16 = torch.cuda.get_device_capability()[0] >= 8      # bfloat16 は Ampere 以降。T4 では float16 を使う
DTYPE = torch.bfloat16 if BF16 else torch.float16
MODEL_NAME = "sbintuitions/sarashina2.2-3b-instruct-v0.1"
tokenizer = load_tokenizer(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE, device_map="auto").eval()
AB_IDS = [tokenizer(c, add_special_tokens=False)["input_ids"][-1] for c in ("A", "B")]

PROMPT_1 = ("次の質問に対する 2 つの回答 A と B を比べ、全体としてより良い回答はどちらかを選び、A か B の 1 文字だけで答えてください。\n\n"
            "# 質問\n{q}\n\n# 回答 A\n{a}\n\n# 回答 B\n{b}")
PROMPT_2 = ("次の質問に対する 2 つの回答 A と B を比べ、より高品質な回答を選んで A か B の 1 文字だけで答えてください。\n"
            "評価基準:\n- 事実の正確さを最優先し、誤りや矛盾を含む回答は低く評価する\n"
            "- 挨拶・前置き・言い換えなどの水増しによる長さは評価に加えず、簡潔で要点を満たす回答を優先する\n\n"
            "# 質問\n{q}\n\n# 回答 A\n{a}\n\n# 回答 B\n{b}")

@torch.no_grad()
def batch_p_a(prompts, bs=16):
    """次のトークンが A である条件付き確率 P(A) / (P(A) + P(B)) をバッチで 1 回順伝播して求める"""
    texts = [tokenizer.apply_chat_template([{"role": "user", "content": p}], add_generation_prompt=True, tokenize=False) for p in prompts]
    out = []
    for i in range(0, len(texts), bs):
        enc = tokenizer(texts[i:i + bs], return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        pos = (enc["attention_mask"].cumsum(-1) - 1).clamp(min=0)
        logits = model(**enc, position_ids=pos, logits_to_keep=1).logits[:, -1, AB_IDS].float()
        out.extend(logits.softmax(-1)[:, 0].cpu().tolist())
    return np.array(out)

pairs_meta = [(qi, s1, s2) for qi in range(len(DATASET)) for s1, s2 in itertools.combinations(SYSTEMS, 2)]
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
raw_probs = {}
for label, tmpl in [("P1_通常", PROMPT_1), ("P2_長さ補正", PROMPT_2)]:
    fwd_prompts = [tmpl.format(q=DATASET[qi]["q"], a=DATASET[qi][s1], b=DATASET[qi][s2]) for qi, s1, s2 in pairs_meta]
    bwd_prompts = [tmpl.format(q=DATASET[qi]["q"], a=DATASET[qi][s2], b=DATASET[qi][s1]) for qi, s1, s2 in pairs_meta]
    raw_probs[label] = (batch_p_a(fwd_prompts), batch_p_a(bwd_prompts))

print(f"800 件の一対比較を {time.time() - t0:.1f} 秒で完了 | GPU メモリ最大 {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
del model
gc.collect(); torch.cuda.empty_cache()

800 件の一対比較を 17.6 秒で完了 | GPU メモリ最大 7.04 GB


In [3]:
# 1. 位置バイアス・長さバイアス・勝率行列の計測
bias_rows = []
for pname, label in [("P1_通常", "P1: 通常の一対比較"), ("P2_長さ補正", "P2: 長さ補正・事実重視")]:
    p_fwd, p_bwd_a = raw_probs[pname]      # p_fwd = P(A 勝ち | (s1, s2)), p_bwd_a = P(A 勝ち | (s2, s1))
    p_s1_fwd = p_fwd
    p_s1_bwd = 1.0 - p_bwd_a
    p_s1_avg = 0.5 * (p_s1_fwd + p_s1_bwd)
    pos_a_win = float(np.r_[p_fwd > 0.5, p_bwd_a > 0.5].mean())
    pos_a_mean = float(np.r_[p_fwd, p_bwd_a].mean())
    cons = float(((p_s1_fwd > 0.5) == (p_s1_bwd > 0.5)).mean())
    idx13 = [k for k, (_, s1, s2) in enumerate(pairs_meta) if s1 == "S1_簡潔正確" and s2 == "S3_冗長不正確"]
    bias_rows.append({
        "プロンプト": label,
        "先頭 (A) 勝率": round(pos_a_win, 3),
        "平均 P(A)": round(pos_a_mean, 3),
        "双方向一致率": round(cons, 3),
        "順序反転率": round(1.0 - cons, 3),
        "簡潔 vs 冗長 勝率": round(float((p_s1_avg[idx13] > 0.5).mean()), 3),
        "簡潔 vs 冗長 平均確率": round(float(p_s1_avg[idx13].mean()), 3),
    })

pd.set_option("display.width", 220); pd.set_option("display.unicode.east_asian_width", True)
print("--- 位置バイアスと長さバイアスの診断 ---")
print(pd.DataFrame(bias_rows).to_string(index=False))

idx_map = {s: i for i, s in enumerate(SYSTEMS)}
def win_matrix(p_arr):
    W = np.zeros((len(SYSTEMS), len(SYSTEMS)))
    for k, (_, s1, s2) in enumerate(pairs_meta):
        i, j = idx_map[s1], idx_map[s2]
        W[i, j] += p_arr[k]
        W[j, i] += 1.0 - p_arr[k]
    mat = W / len(DATASET)
    np.fill_diagonal(mat, np.nan)
    return pd.DataFrame(mat, index=SYSTEMS, columns=SYSTEMS).round(2)

print("\n--- 順序入れ替え平均の勝率行列（行が列に勝つ期待確率）: P1 通常 ---")
print(win_matrix(0.5 * (raw_probs["P1_通常"][0] + (1.0 - raw_probs["P1_通常"][1]))).to_string())
print("\n--- 順序入れ替え平均の勝率行列（行が列に勝つ期待確率）: P2 長さ補正 ---")
print(win_matrix(0.5 * (raw_probs["P2_長さ補正"][0] + (1.0 - raw_probs["P2_長さ補正"][1]))).to_string())

--- 位置バイアスと長さバイアスの診断 ---
            プロンプト  先頭 (A) 勝率  平均 P(A)  双方向一致率  順序反転率  簡潔 vs 冗長 勝率  簡潔 vs 冗長 平均確率
    P1: 通常の一対比較          0.328      0.328         0.660       0.340               0.35                  0.541
P2: 長さ補正・事実重視          0.343      0.343         0.675       0.325               1.00                  0.870

--- 順序入れ替え平均の勝率行列（行が列に勝つ期待確率）: P1 通常 ---
               S1_簡潔正確  S2_標準実用  S3_冗長不正確  S4_曖昧一般論  S5_幻覚逸脱
S1_簡潔正確            NaN         0.55           0.54           0.73         1.00
S2_標準実用           0.45          NaN           0.49           0.71         1.00
S3_冗長不正確         0.46         0.51            NaN           0.78         0.95
S4_曖昧一般論         0.27         0.29           0.22            NaN         0.96
S5_幻覚逸脱           0.00         0.00           0.05           0.04          NaN

--- 順序入れ替え平均の勝率行列（行が列に勝つ期待確率）: P2 長さ補正 ---
               S1_簡潔正確  S2_標準実用  S3_冗長不正確  S4_曖昧一般論  S5_幻覚逸脱
S1_簡潔正確            NaN         0.61           0.87           0.92         0

In [4]:
# 2. Bradley-Terry / Elo レーティングの推定 (base=1000, scale=400) と 1,000 回質問単位ブートストラップ 95% CI
pair_ij = np.array([(idx_map[s1], idx_map[s2]) for _, s1, s2 in pairs_meta])
pair_q = np.array([qi for qi, _, _ in pairs_meta])
human_vals = np.array([HUMAN_SCORE[s] for s in SYSTEMS], dtype=float)

def fit_elo(W, reg=1e-2):
    K = W.shape[0]
    iu, ju = np.triu_indices(K, 1)
    w_ij, w_ji = W[iu, ju], W[ju, iu]
    n_ij = w_ij + w_ji

    def loss_and_grad(theta):
        diff = theta[iu] - theta[ju]
        nll = np.sum(w_ij * np.logaddexp(0, -diff) + w_ji * np.logaddexp(0, diff)) + 0.5 * reg * np.sum(theta ** 2)
        p = 1.0 / (1.0 + np.exp(-np.clip(diff, -30, 30)))
        g_pair = n_ij * p - w_ij
        grad = reg * theta.copy()
        np.add.at(grad, iu, g_pair)
        np.add.at(grad, ju, -g_pair)
        return nll, grad

    opt = minimize(loss_and_grad, np.zeros(K), jac=True, method="BFGS")
    th = opt.x - opt.x.mean()
    return 1000.0 + (400.0 / np.log(10.0)) * th

def evaluate_elo(p_arr, n_boot=1000):
    per_q = np.zeros((len(DATASET), len(SYSTEMS), len(SYSTEMS)))
    for k, (qi, (i, j)) in enumerate(zip(pair_q, pair_ij)):
        per_q[qi, i, j] += p_arr[k]
        per_q[qi, j, i] += 1.0 - p_arr[k]
    pt = fit_elo(per_q.sum(0))
    rng = np.random.default_rng(0)
    boots = np.array([fit_elo(per_q[rng.integers(0, len(DATASET), len(DATASET))].sum(0)) for _ in range(n_boot)])
    return pt, np.percentile(boots, 2.5, axis=0), np.percentile(boots, 97.5, axis=0)

conditions = [
    ("片側順 (A, B) のみ", raw_probs["P1_通常"][0]),
    ("順序入れ替え平均（通常プロンプト）", 0.5 * (raw_probs["P1_通常"][0] + (1.0 - raw_probs["P1_通常"][1]))),
    ("順序入れ替え平均 + 長さ補正ルブリック", 0.5 * (raw_probs["P2_長さ補正"][0] + (1.0 - raw_probs["P2_長さ補正"][1]))),
]
elo_table = {}
corr_rows = []
for cname, p_arr in conditions:
    pt, lo, hi = evaluate_elo(p_arr)
    elo_table[cname] = [f"{e:6.1f} [{l:6.1f}, {h:6.1f}]" for e, l, h in zip(pt, lo, hi)]
    corr_rows.append({
        "条件": cname,
        "Spearman ρ": round(float(spearmanr(human_vals, pt).statistic), 3),
        "Kendall τ": round(float(kendalltau(human_vals, pt).statistic), 3),
    })

print("--- Bradley-Terry / Elo レーティングと 95% ブートストラップ信頼区間 ---")
print(pd.DataFrame(elo_table, index=SYSTEMS).to_string())
print("\n--- 人手参照スコアとの順位相関 ---")
print(pd.DataFrame(corr_rows).to_string(index=False))

--- Bradley-Terry / Elo レーティングと 95% ブートストラップ信頼区間 ---
                    片側順 (A, B) のみ 順序入れ替え平均（通常プロンプト） 順序入れ替え平均 + 長さ補正ルブリック
S1_簡潔正確    1004.7 [ 927.3, 1075.7]            1199.5 [1161.6, 1256.5]               1289.3 [1235.4, 1353.8]
S2_標準実用    1124.5 [1073.7, 1184.1]            1167.7 [1126.3, 1221.3]               1193.0 [1152.3, 1247.0]
S3_冗長不正確  1216.1 [1103.4, 1368.1]            1177.5 [1092.9, 1267.3]                965.6 [ 904.2, 1027.3]
S4_曖昧一般論  1096.3 [1023.9, 1193.6]             998.9 [ 954.1, 1055.9]                875.9 [ 809.3,  930.5]
S5_幻覚逸脱     558.4 [ 356.3,  647.1]             456.4 [ 275.5,  540.7]                676.2 [ 620.4,  720.7]

--- 人手参照スコアとの順位相関 ---
                                 条件  Spearman ρ  Kendall τ
                   片側順 (A, B) のみ         0.3        0.2
   順序入れ替え平均（通常プロンプト）         0.9        0.8
順序入れ替え平均 + 長さ補正ルブリック         1.0        1.0


In [5]:
# 3. 位置バイアス反転例・長さバイアスの罠例の具体表示
print("--- 位置反転の例（P1 通常プロンプト: 順序を入れ替えると勝者が逆転）---")
shown = 0
for k, (qi, s1, s2) in enumerate(pairs_meta):
    pf, pb = raw_probs["P1_通常"][0][k], 1.0 - raw_probs["P1_通常"][1][k]
    if (pf > 0.5) != (pb > 0.5) and abs(pf - pb) > 0.4:
        print(f"  Q{qi:02d} {s1} vs {s2}: (前,後) P={pf:.3f} / (後,前) P={pb:.3f} | {DATASET[qi]['q'][:40]}")
        shown += 1
        if shown >= 5:
            break

print("\n--- 長さバイアスの罠例（S1_簡潔正確 vs S3_冗長不正確: P1 では冗長が勝ち、P2 で簡潔正確が逆転）---")
shown = 0
for k, (qi, s1, s2) in enumerate(pairs_meta):
    if s1 == "S1_簡潔正確" and s2 == "S3_冗長不正確":
        p1 = 0.5 * (raw_probs["P1_通常"][0][k] + (1.0 - raw_probs["P1_通常"][1][k]))
        p2 = 0.5 * (raw_probs["P2_長さ補正"][0][k] + (1.0 - raw_probs["P2_長さ補正"][1][k]))
        if p1 < 0.5 and p2 > 0.5:
            print(f"  Q{qi:02d}: P1={p1:.3f} -> P2={p2:.3f} | {DATASET[qi]['q'][:45]}")
            shown += 1
            if shown >= 5:
                break

--- 位置反転の例（P1 通常プロンプト: 順序を入れ替えると勝者が逆転）---
  Q00 S1_簡潔正確 vs S2_標準実用: (前,後) P=0.000 / (後,前) P=0.893 | Python で辞書から値を取得するとき、`d[k]` と `d.get(k)`
  Q00 S1_簡潔正確 vs S4_曖昧一般論: (前,後) P=0.133 / (後,前) P=1.000 | Python で辞書から値を取得するとき、`d[k]` と `d.get(k)`
  Q00 S2_標準実用 vs S3_冗長不正確: (前,後) P=0.002 / (後,前) P=0.953 | Python で辞書から値を取得するとき、`d[k]` と `d.get(k)`
  Q01 S1_簡潔正確 vs S2_標準実用: (前,後) P=0.148 / (後,前) P=1.000 | HTTP ステータスコードの 404 と 500 の違いを教えてください。
  Q01 S1_簡潔正確 vs S3_冗長不正確: (前,後) P=0.011 / (後,前) P=0.986 | HTTP ステータスコードの 404 と 500 の違いを教えてください。

--- 長さバイアスの罠例（S1_簡潔正確 vs S3_冗長不正確: P1 では冗長が勝ち、P2 で簡潔正確が逆転）---
  Q00: P1=0.123 -> P2=0.873 | Python で辞書から値を取得するとき、`d[k]` と `d.get(k)` はどう違
  Q01: P1=0.498 -> P2=0.990 | HTTP ステータスコードの 404 と 500 の違いを教えてください。
  Q04: P1=0.475 -> P2=0.533 | 日本の消費税における軽減税率 (8%) の対象と標準税率 (10%) になる例を教えてくだ
  Q05: P1=0.007 -> P2=0.957 | 日本の特定商取引法における訪問販売のクーリング・オフ期間は何日間ですか。
  Q06: P1=0.498 -> P2=0.582 | 日本で 110 番と 119 番はそれぞれどのようなときにかけますか。


### 実行結果の分析

- 位置バイアスの大きさ（`sarashina2.2-3b-instruct-v0.1`、800 件の比較を 17.6 秒で完了、GPU メモリ最大 7.04 GB）: 1 回あたりの先頭 `(A)` 勝率は P1 で 0.328（平均 P(A)=0.328）、P2 でも 0.343（平均 P(A)=0.343）にとどまり、後に提示された `B` 側を約 66〜67% の確率で選ぶ強い後置バイアス (recency bias) が現れた。順序 `(A, B)` と `(B, A)` の双方向で同じ勝者を選ぶ一致率は P1 で 0.660（順序反転率 0.340）、P2 で 0.675（同 0.325）であり、約 3 割の比較で順序を入れ替えただけで勝敗が逆転する（例: `Q00 S1_簡潔正確 vs S2_標準実用` は前置き時 `P=0.000`・後置き時 `P=0.893`、`Q01 S1_簡潔正確 vs S3_冗長不正確` は `0.011` と `0.986`）。
- 位置バイアスが Elo 順位を崩す様子: 順序を入れ替えずに固定順 `片側順 (A, B) のみ` で Bradley-Terry モデルを当てはめると、リストの前方に置かれやすい `S1_簡潔正確` は Elo 1004.7 [927.3, 1075.7]（4 位）へ落ち、より後ろに置かれやすい `S3_冗長不正確` (1216.1)・`S2_標準実用` (1124.5)・`S4_曖昧一般論` (1096.3) を下回って、人手参照スコアとの Spearman ρ は 0.3 (Kendall τ=0.2) まで崩れた。
- 長さバイアスの定量化とルーブリック補正の効果: `S3_冗長不正確` は平均 170.4 文字（`S1_簡潔正確` 69.0 文字の約 2.5 倍）で丁寧な前置きを備える一方、中核の事実に誤りを含む。位置バイアスを打ち消す `順序入れ替え平均（通常プロンプト）` にすると `S1_簡潔正確` が 1199.5 [1161.6, 1256.5] で 1 位へ戻るものの、`S1 vs S3` の直接対決で `S1` の勝率は 0.35（平均確率 0.541）にとどまり、Elo でも `S3_冗長不正確` (1177.5 [1092.9, 1267.3]) が `S2_標準実用` (1167.7 [1126.3, 1221.3]) を上回って ρ=0.9 (τ=0.8) になった。これに対し、事実の正確さ優先と水増し除外を明記した `順序入れ替え平均 + 長さ補正ルブリック` では、`S1 vs S3` の勝率が 0.35 → 1.00（平均確率 0.541 → 0.870）へ改善し（例: `Q05` は 0.007 → 0.957、`Q00` は 0.123 → 0.873）、Elo は `S1` 1289.3 [1235.4, 1353.8] ＞ `S2` 1193.0 [1152.3, 1247.0] ＞ `S3` 965.6 [904.2, 1027.3] ＞ `S4` 875.9 [809.3, 930.5] ＞ `S5` 676.2 [620.4, 720.7] と人手参照スコアに完全一致 (ρ=1.0, τ=1.0) した。`S1`・`S2` と `S3` の 95% 信頼区間も重ならなくなった。

### 実務上の指針

- 一対比較の LLM-as-a-judge では「順序の入れ替え（両方向平均）」を必須とし、先頭勝率と双方向一致率を必ずログに出す。順序反転率が 3 割前後ある評価器では、片側順だけの A/B テストはモデルの真の実力ではなく「A と B のどちらのスロットに入れたか」を測ってしまう。
- 順序入れ替えは位置バイアスは消せるが、長さバイアスや丁寧語への好意バイアスまでは消せない。プロンプトに「前置き・挨拶・冗長な言い換えを評価しない」「事実誤認を厳しく減点する」と明記し、長くて誤りを含むダミー応答に対する勝率を定点観測する。
- Elo の信頼区間は、全ペアを独立にリサンプリングするのではなく、このレシピのように「質問（プロンプト）単位」で復元抽出して求める。同じ質問を共有する $\binom{K}{2}$ 組の勝敗は独立でないため、ペア単位のリサンプリングでは信頼区間が不当に狭くなる。また勝率 100%・0% に近いシステムが含まれても発散しないよう、Bradley-Terry 対数尤度に微小な L2 正則化（ここでは `reg=1e-2`）を入れて安定化する。
- ここでの 3B モデルによる 20 問は、位置・長さバイアスの診断と Elo 算出手順を示すための小規模検証である。本番のモデル順位付けでは数百問以上を用意し、自己選好バイアス（評価器と同系統のモデルを高く評価する傾向）を避けるため、評価対象とは別系統・より高性能なモデルや人手の部分監査を併用する。

### 関連レシピ

- 点数付け（1〜5 の期待値）と「はい / いいえ」分解、および順序入れ替え一対比較の基本は、この章の「手元の LLM に生成文を採点させる (LLM-as-a-judge)」および「LLM に生成文の品質を採点させる (LLM-as-a-judge)」を参照。
- 一対比較データから応答ごとのスカラー報酬を学習する仕組みと、長さに引っ張られる報酬モデルの性質は、LLM の学習と調整の章の「選好データから報酬モデルを学習して応答を採点する（Bradley-Terry モデル）」および「報酬モデルで N 個の応答を採点して最良を選ぶ (Best-of-N)」を参照。
- ブートストラップ法と順位相関係数（Spearman の ρ、Kendall の τ）の解説は、評価指標と統計的検定の章の「ブートストラップ法で評価値の信頼区間を求める」および「自動評価指標と人手評価の相関を調べる」を参照。
- `sarashina2.2` のトークナイザー Unigram 復元については、大規模言語モデルの章の「オープンな LLM とチャットする（チャットテンプレート）」を参照。
- 順位評価指標（Spearman の ρ・Kendall の τ）の定義や Pairwise ランキングと Pointwise 回帰の比較演習は、姉妹編『[自然言語処理の教科書](https://github.com/mamorlis/nlpbook)』第 2 章の [ランキング問題と順位評価指標（Kendall's τ・Spearman's ρ・nDCG）](https://github.com/mamorlis/nlpbook/blob/main/notebooks/ch2_ranking.ipynb) を、プロンプトによる LLM 評価とハルシネーション抑制の演習は第 4 章の [軽量オープン LLM によるプロンプト演習とハルシネーション抑制](https://github.com/mamorlis/nlpbook/blob/main/notebooks/ch4_llm_prompt.ipynb) も参照。